<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/SentenceTransformers/notebooks/1_sent_transf_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Text embeddings with Sentence Transformers

**A Methods Workshop in Computational Text Analysis**
- Art Dewulf · Public Administration and Policy Group · Wageningen University and Research
- Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

Sentence Transformers leverage contextual embeddings from BERT or GPT models to convert sentences or larger chunks of text into embedding vectors. Sentence Transformers are trained such that text chunks that are similar in semantic meaning are converted to embedding vectors that are close to each other in the vector space (often captured through cosine similarity). Sentence transformer models can be used for a variety of NLP tasks, including retrieval, deduplication, keyword extraction, classification and topic modeling.

We illustrate some of these applications in this notebook:
- text retrieval
- deduplication
- keyword extraction
- sentiment analysis
- text classification

> **Important — turn on the GPU!** In Colab: *Runtime → Change runtime type → T4 GPU*.
> Sentence Transformers are slow on CPU, although smaller models can also run on CPU.


# Preliminaries


In [ ]:
# sentence_transformers library for converting text into embedding vectors
from sentence_transformers import SentenceTransformer

# cosine_similarity metric from sklearn library to compare embedding vectors
from sklearn.metrics.pairwise import cosine_similarity

# datasets
import datasets

# pandas library for DataFrames
import pandas as pd

# numpy library for arrays (matrices)
import numpy as np

# matplotlib library for plotting
from matplotlib import pyplot as plt

# Check whether a GPU is available — this makes Sentence Transformers much faster
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device.upper())
if device == "cpu":
    print("⚠️  No GPU detected. Things will be slow. "
          "Consider: Runtime → Change runtime type → T4 GPU.")

# Load Sentence Transformer model

- for most of the Sentence Transformer models, you need to create an account on https://huggingface.co
- we use the model `embeddinggemma-300m`, for which you need to accept the license, so go to https://huggingface.co/google/embeddinggemma-300m and click on the license
- in your Huggingface account settings, go to Access Tokens, and create a token with the name HF_TOKEN, and when you see the token (a long list of letters) copy it because it will only be shown once
- in this Google Colab go to the Secrets tab, and Add new secret, with the name HF_TOKEN and the value (the long list of letters) from Huggingface
- finally, check the switch that says Notebook access, to give this Colab notebook access to the token

This Sentence Transformer model generates 768-dimensional embedding vectors, which means that each text chunk will be transformed in to a list of 768 numbers, which are like coordinates in a 768-dimensional space. Each text chunk becomes a point in this multidimensional space, and the model is trained such that text chunks with highly similar semantic meaning end up close to each other in this space, while chunks with very different different semantic meaning end up far away from each other in the space.

In [ ]:
# load Sentence Transformer model embeddinggemma-300m, a 300 million parameter model
# that can be applied to texts of maximum 2048 tokens, generating embedding vectors
# with 768 dimensions
model = SentenceTransformer('google/embeddinggemma-300m')
model

# Embeddings and cosine similarity

In [ ]:
# create a list of text chunks
chunks = ["World Food Programme Warns Middle East Crisis Driving Developing Countries Into Hunger",
        "Iran war Pushes Vulnerable Nations Toward Acute Food Insecurity",
        "Nitrogen crisis in the Dutch agricultural sector leads to farmer protests"]

In [ ]:
# use the model to get the embedding vectors for these three documents
embs = model.encode(chunks, show_progress_bar=True)
# the result is a 2-dimensionsal numpy array (i.e. a matrix) with 3 rows (the 3 embedding vectors) which are each 768 columns long, so shape (3, 768)
embs.shape

In [ ]:
# visualize the array (basically a list of rows, where each row is a list of values)
embs

In [ ]:
# print the number of vectors, the length of the first embedding vector, and the first embedding vector itself
print(len(embs), 'vectors')
print(len(embs[0]), 'dimensions')
print('first embedding vector:')
embs[0]

In [ ]:
# create a small function to plot the values of an embeddings vector
def plot_embedding(embedding):
  pd.Series(embedding).plot.bar(figsize=(32,2), grid=False)

In [ ]:
# plot the three embedding vectors
for i, emb in enumerate(embs):
  print(f'embedding vector for "{chunks[i]}"')
  plot_embedding(emb)
  plt.tick_params(axis='x', labelbottom=False)
  plt.show()
  print()

In [ ]:
# calculate the cosine similarity between each pair of these three embedding vectors
sim_matrix = cosine_similarity(embs)
sim_matrix

In [ ]:
# visualize the similarities between the embedding vectors of the three documents
pd.DataFrame(sim_matrix, index=chunks, columns=chunks).round(2)

> **✏️ Exercise**
>
> Replace the third sentence with your own sentence, and try to guess what the similarities will look like.

# Application: text retrieval

Text searching often goes beyond searching literal text, and text embeddings are a common way to enhance information retrieval. The cosine similarity between the query and a list of texts can rank these texts in terms of relevance, even when the literal words of the query do not appear in the texts.

In [ ]:
# 10 short example sentences (corpus)
corpus = [
    "Rising sea levels threaten coastal cities worldwide.",
    "Heavy rainfall causes severe flooding in urban areas.",
    "Prolonged droughts reduce agricultural yields significantly.",
    "Melting glaciers contribute directly to sea level rise.",
    "Water scarcity is a growing problem in arid regions.",
    "Climate change intensifies extreme weather events.",
    "Groundwater depletion affects local freshwater supplies.",
    "Ocean acidification harms delicate coral reef ecosystems.",
    "Sustainable water management is crucial for the future.",
    "Warmer global temperatures increase evaporation rates.",
]

# A query sentence
query = "How does global warming affect ocean water levels?"

# Encode the corpus and the query
corpus_embeddings = model.encode(corpus)
query_embedding = model.encode([query])

# Calculate cosine similarities
similarities = cosine_similarity(query_embedding, corpus_embeddings)[0]

# Create a Series with the sentences as index, sort by similarity descending
ranking = pd.Series(similarities, index=corpus).sort_values(ascending=False)
ranking

> **✏️ Exercise**
>
> Add a sentence to the corpus list and rerun the code.
> Try to make your new sentence end up as the most
> similar sentence.

# Application: deduplication

Filtering out near-duplicates from a dataset of texts can be done on the basis of cosine similarity and a threshold.

In [ ]:
# short sentences with a semantic near-duplicate
texts = [
    "Rising sea levels threaten coastal cities.",
    "Coastal urban areas are at risk from rising oceans.", # Near-duplicate
    "Heavy rainfall causes severe flooding.",
    "Prolonged droughts reduce agricultural yields.",
    "Melting glaciers contribute to sea level rise.",
    "Water scarcity is a growing problem in arid regions.",
    "Climate change intensifies extreme weather events.",
    "Ocean acidification harms coral reef ecosystems."
]

# Encode the texts
embeddings = model.encode(texts)

# Calculate cosine similarity matrix
sim_matrix = cosine_similarity(embeddings)

# Show similarities
simdf = pd.DataFrame(sim_matrix, index=texts, columns=texts).round(4)
simdf

In [ ]:
simdf_unstacked = simdf.unstack().sort_values(ascending=False)
simdf_unstacked = simdf_unstacked[simdf_unstacked < 1].drop_duplicates()
simdf_unstacked.head(5)

In [ ]:
threshold = 0.8
simdf_unstacked[simdf_unstacked > threshold]

> **✏️ Exercise**
>
> Add two sentences that are near duplicates, and try
> to set the threshold at the right level to filter
> them out.

# Application: keyword extraction

In [ ]:
# Define a long example sentence
sentence = '''Rivers naturally meander through valleys, causing erosion and
transporting sediment, gradually reshaping the entire landscape over
thousands of years into intricate networks.'''

# Extract unique words (with basic preprocessing to remove punctuation and make lowercase)
words = list(set(word.strip('.,').lower() for word in sentence.split()))

# Encode the entire sentence and the individual words separately
sentence_embedding = model.encode([sentence])
word_embeddings = model.encode(words)

# Calculate cosine similarity between the full sentence and each word
similarities = cosine_similarity(sentence_embedding, word_embeddings)[0]

# Create a Series to rank the words by similarity
keyword_rankings = pd.Series(similarities, index=words).sort_values(ascending=False)

print("Top extracted keywords:")
print(keyword_rankings.head(5))

> **✏️ Exercise**

> Copy a longer piece of text from a website, and extract the top 10 keywords from it

In [ ]:
text = '''
Your text here
'''

# add code to extract keywords

# Application: sentiment analysis

In [ ]:
drought_sentences = [
    "The drought has completely destroyed this year's harvest.",
    "We are hoping for rain soon to end this terrible drought.",
    "The drought-resistant crops are actually thriving despite the lack of rain.",
    "Water restrictions have been implemented due to the ongoing drought.",
    "It's fascinating to see how the local ecosystem adapts to the drought conditions.",
    "The severe drought is causing widespread panic among the farmers.",
    "Thanks to the new reservoir, we survived the drought without any water shortages.",
    "The long drought has finally come to an end with today's heavy rainfall!",
    "A drought is officially defined as a prolonged period of abnormally low rainfall.",
    "Sadly, the prolonged drought has dried up the village well."
]

In [ ]:
text_embeddings = model.encode(drought_sentences)
text_embeddings.shape

In [ ]:
sentiment_categories = ['Positive', 'Negative']

In [ ]:
sentiment_categories_emb = model.encode(sentiment_categories)
sentiment_categories_emb.shape

In [ ]:
sentiment_categories_similarities = cosine_similarity(text_embeddings, sentiment_categories_emb)
sentiment_categories_similarities

In [ ]:
sentdf = pd.DataFrame(sentiment_categories_similarities, index=drought_sentences, columns=sentiment_categories)
sentdf

In [ ]:
sentdf['sentiment_score'] = sentdf['Positive'] - sentdf['Negative']
sentdf['sentiment_score'].sort_values(ascending=False)

> **✏️ Exercise**


> Analyse the sentiment in these sentences, but focus on 'hope' versus 'despair' instead of just 'positive' and 'negative. Store the result in new column called `hope_vs_despair_score`

In [ ]:
# Your code here

drought_sentences = [
    "The drought has completely destroyed this year's harvest.",
    "We are hoping for rain soon to end this terrible drought.",
    "The drought-resistant crops are actually thriving despite the lack of rain.",
    "Water restrictions have been implemented due to the ongoing drought.",
    "It's fascinating to see how the local ecosystem adapts to the drought conditions.",
    "The severe drought is causing widespread panic among the farmers.",
    "Thanks to the new reservoir, we survived the drought without any water shortages.",
    "The long drought has finally come to an end with today's heavy rainfall!",
    "A drought is officially defined as a prolonged period of abnormally low rainfall.",
    "Sadly, the prolonged drought has dried up the village well."
]

text_embeddings = model.encode(drought_sentences)
text_embeddings.shape



# Application: text classification

## Loading and processing the dataset

We use a small dataset of US bill titles, which are labeled according the policy_area the pertain to. We are going to try to predict the policy_area for each title using text embeddings.

In [ ]:
# load the dataset from GitHub
url = 'https://raw.githubusercontent.com/denisejroth/bags-vectors-transformers/main/SentenceTransformers/notebooks/bills.csv'
df = pd.read_csv(url)
df

In [ ]:
# store the policy_area labels as categories
categories = sorted(df['policy_area'].unique())
categories

In [ ]:
# embed the text column
embeddings = model.encode(df['text'].to_list(), batch_size=32, show_progress_bar=True)
embeddings.shape

In [ ]:
df['emb'] = list(embeddings)
df

In [ ]:
# Train/test split
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["policy_area"])
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print('train dataset of size', len(train_df), '- test dataset of size' ,len(test_df))

## Classifying the texts

Text classification with text embeddings works through comparing the embedding of each text, with an embedding vector that represents the category (the category embedding). The text gets classified into the category with the highest cosine similarity to the text.

In [ ]:
# create function to classify a Series of text embeddings, using a list of category embeddings
def classify(embseries, categories, category_embeddings, model):
  '''calculates cosine similarity of each embedding in a Series of text embeddings (embseries)
  with each category in category_embeddings, and returns a 2-column DataFrame with the predicted
  category and the similarity value for each text embeddings, using a Sentence Transformer model'''
  category_similarities = cosine_similarity(np.array(embseries.to_list()), category_embeddings)
  categorydf = pd.DataFrame(category_similarities, columns=categories)
  categorydf['category_pred'] = categorydf[categories].idxmax(axis=1)
  categorydf['category_sim'] = categorydf[categories].max(axis=1)
  return categorydf[['category_pred', 'category_sim']]

### Classification option 1: embed category labels as classification anchors

With this zero-shot type classification we can reasonable classification results (depending on the label formulation and the dataset), but usuall not a very high classification accuracy.

In [ ]:
# get embeddings of the labels or categories
print(categories)
categories_emb = model.encode(categories)
categories_emb.shape

In [ ]:
# use classify function to apply classification to the test_df embeddings
test_df[['ZS_category_pred', 'ZS_category_sim']] = classify(test_df['emb'], categories, categories_emb, model)
test_df.ZS_category_pred.value_counts()

In [ ]:
# check accuracy of classification
from sklearn import metrics
print('classification accuracy', metrics.accuracy_score(test_df.policy_area, test_df.ZS_category_pred))

In [ ]:
# check all classification metrics (f1-score macro-average is an important one)
from sklearn import metrics
print(metrics.classification_report(test_df.policy_area, test_df.ZS_category_pred))

### Classification option 2: use average category embeddings from train set as classification anchors

Classification accuracy can be significantly enhanced by averaging text embeddings for each category from a training set, and using those as anchors for classifying the texts: each text goes into the category with the most similar average category embedding.

In [ ]:
# create category embeddings by averaging text vectors per category in the training dataset
categories_mean_emb = np.array(train_df.groupby('policy_area')['emb'].mean().to_list())
categories_mean_emb.shape

In [ ]:
# use classify function to apply classification to the test_df embeddings
test_df[['mean_category_pred', 'mean_category_sim']] = classify(test_df['emb'], categories, categories_mean_emb, model)
test_df.mean_category_pred.value_counts()

In [ ]:
# check accuracy of classification
from sklearn import metrics
print('classification accuracy', metrics.accuracy_score(test_df.policy_area, test_df.mean_category_pred))

In [ ]:
# check all classification metrics (f1-score macro-average is an important one)
from sklearn import metrics
print(metrics.classification_report(test_df.policy_area, test_df.mean_category_pred))

> **✏️ Exercise**


> A very small training dataset can already be sufficient to significantly enhance classification accuracy, compared to just embedding the label categories. What matters most is the representativeness of the samples in the training dataset - they can even be generated if necessary. Find the code block where the size of the training dataset is defined, change it such that the training dataset is only 10% of the dataset, rerun the classification and check the accuracy. How high are the accuracy scores now for both options? What has changed and why? Here we plot the entire dataset.

## Bonus: visualize embeddings and category centroids

The logic of categorizing each text into the nearest category can be visualized by plotting all the texts, as well as the average category embeddings, as points in a 2-dimensional space. Dimensionality reduction tools like UMAP can be used to reduce the 768-dimensional representation of each text to 2 dimensions, while trying to make sure that text embeddings that are close together (high cosine similarity) in the high-dimensional space are also close together (high cosine similarity) in the 2-dimensional space.

In [ ]:
# use UMAP for dimensionality reduction of the 768-dimensional vectors to 2 dimensions (x and y)
from umap import UMAP
umapper2D = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine', unique=True, verbose=True)
umap_embeddings2D = umapper2D.fit_transform(embeddings)
umap_embeddings2D.shape

In [ ]:
# store x and y coordinates in the DataFrame
df[['x', 'y']] = umap_embeddings2D

In [ ]:
# add numerical representation of the policy area (for coloring the texts according to category)
policy_area_dict = {cat: i for i, cat in enumerate(categories)}
df['policy_area_nr'] = df.policy_area.map(policy_area_dict)

In [ ]:
# make a scatter plot of all texts, coloured by policy_area, and with the category centroids marked as X
ax = df.plot.scatter('x', 'y', c='policy_area_nr', cmap='coolwarm', alpha=0.5, figsize=(24,16))
for idx, row in df.groupby('policy_area')[['x', 'y']].mean().iterrows():
  ax.annotate('X ' + idx, (row['x'], row['y']), fontsize=12, color='black')

# Wrap-up

You have now worked with a Sentence Transformer model and explored a number of different applications of this "Swiss army knife" of computational text analysis.

### Where to go next

- Go through the notebook on topic modelling with text embeddings for a more elaborate application
- Browse the **Hugging Face Hub** for Sentence Transformer models: choose Models/sentencetransformer
- Always keep the **Day 1 discipline**: a held-out test set, and an honest baseline to beat.
